In [ ]:
import sys
import os

# 1. DRIVE MOUNT AND PATHS SETUP
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory: {os.getcwd()}")
except ImportError:
    # Fallback for local execution
    DRIVE_ROOT = '.'
    print("Local execution")

In [ ]:
import os
import json
import glob
import re
from collections import Counter, defaultdict
from datetime import timedelta

BASE_DIR = "/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT"
DETECTION_DIR = os.path.join(BASE_DIR, "video_general_obj_det_finished")
ACTION_DIR = os.path.join(BASE_DIR, "action_recognition_finished")
OUT_CSVS = os.path.join(BASE_DIR, "Video_Summary_CSVs")

os.makedirs(OUT_CSVS, exist_ok=True)

video_to_action_files = defaultdict(list)
for act_file in glob.glob(os.path.join(ACTION_DIR, "*_actions.json")):
    basename = os.path.basename(act_file)
    match = re.search(r'(.*)_person_(\d+)_actions\.json', basename)
    if match:
        video_id = match.group(1)
        video_to_action_files[video_id].append((act_file, match.group(2)))

for video_id, act_files in video_to_action_files.items():
    det_folder = os.path.join(DETECTION_DIR, video_id)
    det_files = glob.glob(os.path.join(det_folder, "*.json"))
    if not det_files:
        continue

    with open(det_files[0], 'r') as f:
        det_data = json.load(f)

    objects = det_data.get("detected_objects", {})
    video_rows = []

    for act_file, person_id in act_files:
        person_key = f"person_{person_id}"
        if person_key not in objects:
            continue

        bboxes = objects[person_key].get("bbox", [])

        with open(act_file, 'r') as f:
            actions_data = json.load(f)

        action_frequencies = Counter()
        windows = []

        for frame_range, action_info in actions_data.items():
            if isinstance(action_info, list):
                act_list = action_info
            elif isinstance(action_info, dict) and "action" in action_info:
                act_list = [action_info["action"]]
            else:
                continue

            if not act_list:
                continue

            act_str = "-".join(act_list)

            rmatch = re.search(r'frames_(\d+)_to_(\d+)', frame_range)
            if not rmatch:
                continue

            start_f, end_f = int(rmatch.group(1)), int(rmatch.group(2))
            action_frequencies[act_str] += 1
            windows.append((start_f, end_f, act_str))

        for frame_idx in range(0, len(bboxes), 30):
            bbox = bboxes[frame_idx]
            if not bbox:
                continue

            candidate_actions = [act for sf, ef, act in windows if sf <= frame_idx <= ef]
            if not candidate_actions:
                continue

            best_action = max(candidate_actions, key=lambda a: action_frequencies[a])

            frame_objects = []
            for obj_key, obj_data in objects.items():
                if obj_data.get("class_name") != "person":
                    obj_bboxes = obj_data.get("bbox", [])
                    if frame_idx < len(obj_bboxes) and obj_bboxes[frame_idx]:
                        clean_obj_key = re.sub(r'[\s_]+', '_', obj_key)
                        frame_objects.append(clean_obj_key)

            objects_str = "-".join(frame_objects) if frame_objects else "Nothing"

            seconds = frame_idx // 30
            time_str = str(timedelta(seconds=seconds))
            if len(time_str.split(':')) == 3 and len(time_str.split(':')[0]) == 1:
                time_str = "0" + time_str

            video_rows.append({
                "frame": frame_idx,
                "time": time_str,
                "action": best_action,
                "person": person_key,
                "object": objects_str
            })

    if not video_rows:
        continue

    video_rows.sort(key=lambda x: (x["frame"], x["person"]))

    csv_path = os.path.join(OUT_CSVS, f"{video_id}.csv")
    with open(csv_path, 'w') as f:
        f.write("Time,Action,person,Object\n")
        for row in video_rows:
            f.write(f"{row['time']},{row['action']},{row['person']},{row['object']}\n")

print(f"File CSV generati con successo nella cartella: {OUT_CSVS}")